# Smriti × LongMemEval — **Real SVO Algorithm (Llama 3 8B)**

This notebook runs the **full Smriti logic** including AI-powered Subject-Verb-Object (SVO) extraction using a local Llama-3-8B-Instruct model via LLM on Kaggle's free T4 GPUs. This tests the real-world performance of Smriti.

> **Requirements:**
> - Kaggle Accelerator: **GPU T4 x2**
> - Dataset: Add your longmemeval-clean dataset to this notebook.
> - **HuggingFace Token**: You must add a Kaggle Secret named HF_TOKEN with a read-access HuggingFace token, and accept the Llama 3 license on HuggingFace.


In [ ]:
# ── 1. Install Dependencies ───────────────────────────────────────────────────
!pip install -q fastapi uvicorn[standard] httpx sentence-transformers chromadb vllm transformers huggingface_hub


In [ ]:
# ── 2. Authenticate HuggingFace ───────────────────────────────────────────────
import os
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(hf_token)
    print("Successfully logged into HuggingFace!")
except Exception as e:
    print("⚠️ WARNING: Could not load HF_TOKEN from Kaggle Secrets.")
    print("Make sure you added the secret via Add-ons > Secrets in the top menu.")
    print(e)


In [ ]:
# ── 3. Configuration & Paths ──────────────────────────────────────────────────
DATA_DIR        = '/kaggle/input/longmemeval-clean/data/longmemeval'
S_SPLIT_PATH    = os.path.join(DATA_DIR, 'longmemeval_s_cleaned.json')
M_SPLIT_PATH    = os.path.join(DATA_DIR, 'longmemeval_m_cleaned.json')
OUTPUT_DIR      = '/kaggle/working'

GAP_THRESHOLD         = 0.08
MAX_CUTOFF            = 0.52
SIMILARITY_THRESHOLD  = 0.85
SERVER_PORT           = 8976


In [ ]:
# ── 4. Initialize Local vLLM Llama-3 Engine ───────────────────────────────────
import vllm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load Llama-3 on both T4 GPUs via tensor parallelism
llm = LLM(
    model=MODEL_ID, 
    tensor_parallel_size=2, 
    dtype="half", 
    max_model_len=4096, 
    gpu_memory_utilization=0.90,
    enforce_eager=True  # Recommended for Kaggle T4s
)

sampling_params = SamplingParams(temperature=0.1, max_tokens=1000)


In [ ]:
# ── 5. Real Smriti SVO Parsing Logic ──────────────────────────────────────────
import re
import json

SVO_PROMPT = """You are a structured event extractor for the Chronos temporal memory system.

Given the following text, extract ALL Subject-Verb-Object (SVO) events with timestamps.

Rules:
1. Each event must have: subject (who/what), verb (action), object (target/recipient).
2. If a timestamp is mentioned or implied, include it. Otherwise use "now".
3. Return ONLY a valid JSON array — no markdown, no explanation.

Output format (JSON array):
[
  {
    "subject": "string",
    "verb": "string",
    "object": "string",
    "timestamp": "ISO 8601 datetime string"
  }
]

Text to analyze:
---
{text}
---

Extract all SVO events as JSON:"""

def extract_svo_batch(texts):
    prompts = []
    for t in texts:
        messages = [
            {"role": "system", "content": "You are a precise JSON event extractor. Output ONLY a valid JSON array. No markdown, no explanation."},
            {"role": "user", "content": SVO_PROMPT.format(text=t)}
        ]
        # Use tokenizer to apply Llama-3 chat template
        p = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(p)
    
    # Generate in parallel
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    
    results = []
    for out in outputs:
        raw = out.outputs[0].text
        cleaned = raw.strip()
        if cleaned.startswith("`"):
            cleaned = re.sub(r"^`(?:json)?\s*", "", cleaned)
            cleaned = re.sub(r"\s*`$", "", cleaned).strip()
        try:
            array_match = re.search(r"\[.*\]", cleaned, re.DOTALL)
            events = json.loads(array_match.group(0)) if array_match else []
        except:
            events = []
        results.append(events)
    return results


In [ ]:
# ── 6. In-Memory Stores ───────────────────────────────────────────────────────
import uuid
from dataclasses import dataclass
from typing import List, Dict, Any

@dataclass
class EventRecord:
    id: str
    subject: str
    verb: str
    object: str
    raw_text: str

class InMemoryMemoryStore:
    def __init__(self):
        self.events: Dict[str, EventRecord] = {}

class InMemoryVectorStore:
    def __init__(self):
        import chromadb
        from sentence_transformers import SentenceTransformer
        self._model  = SentenceTransformer('all-MiniLM-L6-v2')
        self._chroma = chromadb.Client()
        self._col    = self._chroma.get_or_create_collection('smriti_bench', metadata={'hnsw:space': 'cosine'})

    def store_embeddings(self, records: List[EventRecord]):
        if not records: return
        # Real Smriti logic: Embed "Subject Verb Object | raw_text"
        texts = [f"{r.subject} {r.verb} {r.object} | {r.raw_text}" for r in records]
        embeddings = self._model.encode(texts, normalize_embeddings=True).tolist()
        self._col.add(
            ids=[r.id for r in records],
            embeddings=embeddings,
            documents=[r.raw_text for r in records] # We return the raw text to the scorer
        )

    def semantic_search(self, query: str, similarity_threshold=0.85, n_results=10):
        if self._col.count() == 0: return []
        q_emb = self._model.encode([query], normalize_embeddings=True).tolist()
        res = self._col.query(query_embeddings=q_emb, n_results=min(n_results, self._col.count()))
        out = []
        for i, (doc_id, dist) in enumerate(zip(res['ids'][0], res['distances'][0])):
            if dist <= (1 - similarity_threshold):
                out.append({'id': doc_id, 'distance': dist, 'text': res['documents'][0][i]})
        return out


In [ ]:
# ── 7. Smriti Harness Logic (Gap Cutoff + Filter) ─────────────────────────────
store = InMemoryMemoryStore()
vs    = InMemoryVectorStore()

_STOPWORDS = {'the','a','an','is','in','of','to','it','was','has','are','and','or','for','with','on','at','by','from','this','that','not','but','its','be','as','can','all','use','also','via','per','when','than'}
def extract_entities(text: str) -> set:
    return set(re.findall(r'\b[a-zA-Z0-9_-]{3,}\b', text.lower())) - _STOPWORDS

def bayesian_gap_cutoff(distances):
    if not distances: return MAX_CUTOFF
    sd = sorted(distances)
    if len(sd) > 1 and (sd[1] - sd[0]) > GAP_THRESHOLD:
        return sd[0] + 0.04
    return min(MAX_CUTOFF, sd[0] + 0.12)

def recall(query: str, max_k: int = 5):
    raw = vs.semantic_search(query, similarity_threshold=SIMILARITY_THRESHOLD, n_results=max_k)
    if not raw: return []
    
    q_ents = extract_entities(query)
    filtered = []
    for r in raw:
        evt = store.events.get(r['id'])
        if evt and len(q_ents) >= 2:
            doc_ents = extract_entities(f"{evt.subject} {evt.object} {evt.raw_text}")
            if not (q_ents & doc_ents): continue
        filtered.append(r)
    candidates = filtered if filtered else raw
    
    dists = [r['distance'] for r in candidates]
    cutoff = bayesian_gap_cutoff(dists)
    return [r for r in candidates if r['distance'] <= cutoff]


In [ ]:
# ── 8. Eval Engine ────────────────────────────────────────────────────────────
import time as time_mod
from collections import defaultdict
def substring_match(pred: str, gold: str) -> float:
    return 1.0 if gold.lower().strip() in pred.lower() else 0.0

def tok(text: str):
    return set(re.sub(r'[^a-z0-9\s]', '', text.lower()).split())

def f1_score(pred: str, gold: str) -> float:
    p_toks, g_toks = tok(pred), tok(gold)
    if not p_toks or not g_toks: return 0.0
    common = p_toks & g_toks
    if not common: return 0.0
    p, r = len(common) / len(p_toks), len(common) / len(g_toks)
    return 2 * p * r / (p + r)

def run_real_eval(cases, label='S'):
    print(f'\n{"="*60}\n  Running Real Smriti (Llama-3) LongMemEval-{label}\n{"="*60}')
    results, latencies, cat_scores = [], [], defaultdict(list)
    
    for i, case in enumerate(cases):
        cid = case['id']
        question, gold, category = case['question'], case['answer'], case['category']
        
        # 1. Reset for new case
        store.events.clear()
        try:
            vs._chroma.delete_collection('smriti_bench')
        except:
            pass
        vs._col = vs._chroma.get_or_create_collection('smriti_bench', metadata={'hnsw:space': 'cosine'})

        # 2. Extract texts
        texts = []
        for session in case['haystack_sessions']:
            if isinstance(session, list):
                for turn in session:
                    content = turn.get('content', '') if isinstance(turn, dict) else str(turn)
                    if content.strip(): texts.append(content[:2000])
            elif isinstance(session, dict):
                content = session.get('content', '')
                if content.strip(): texts.append(content[:2000])

        # 3. AI SVO Extraction (BATCHED VIA VLLM)
        batch_size = 50
        records = []
        for chunk_start in range(0, len(texts), batch_size):
            chunk = texts[chunk_start:chunk_start + batch_size]
            svo_batches = extract_svo_batch(chunk)
            for text, svos in zip(chunk, svo_batches):
                if not svos:
                    records.append(EventRecord(id=str(uuid.uuid4()), subject=text[:50], verb='is', object=text[50:150], raw_text=text))
                else:
                    for s in svos:
                        records.append(EventRecord(id=str(uuid.uuid4()), subject=s.get('subject',''), verb=s.get('verb',''), object=s.get('object',''), raw_text=text))
                        
        for r in records:
            store.events[r.id] = r
        vs.store_embeddings(records)

        # 4. Query & Score
        t0 = time_mod.time()
        res = recall(question, max_k=5)
        lat = (time_mod.time() - t0) * 1000
        latencies.append(lat)
        
        retrieved_text = ' '.join(r['text'] for r in res)
        sub, f1 = substring_match(retrieved_text, gold), f1_score(retrieved_text, gold)
        results.append({'id': cid, 'sub': sub, 'f1': f1, 'lat': lat})
        cat_scores[category].append({'sub': sub, 'f1': f1})
        
        if (i+1) % 5 == 0 or i == 0:
            r_sub = sum(r['sub'] for r in results) / len(results)
            print(f'  [{i+1:4d}/{len(cases)}] Sub={r_sub:.3f} lat={lat:.0f}ms')

    n = len(results)
    total_sub = sum(r['sub'] for r in results) / n
    print(f'\nFINAL SCORE: {total_sub*100:.1f}% Recall')
    return results


In [ ]:
# ── 9. Run the Benchmark (S and M Splits)! ────────────────────────────────────
print('Loading datasets...')
with open(S_SPLIT_PATH, 'r', encoding='utf-8') as f:
    cases_s = json.load(f)
with open(M_SPLIT_PATH, 'r', encoding='utf-8') as f:
    cases_m = json.load(f)

print(f"\n--- RUNNING SMALL (S) SPLIT ({len(cases_s[:500])} cases) ---")
results_s = run_real_eval(cases_s[:500], label='S')

print(f"\n--- RUNNING MEDIUM (M) SPLIT ({len(cases_m[:500])} cases) ---")
results_m = run_real_eval(cases_m[:500], label='M')

print("\nAll evaluations complete! Check the output logs for FINAL SCORE.")
